# EVT (padronizado)


*By Miguel Ferreira*

**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo:**
1. Importação do dataset e bibliotecas
2. Execução do ```setup()``` e alinhamento temporal
3. Construção da ferramenta de risco 
4. Sanity checks mínimos
5. DataFrame final (features de risco) 
6. Padronização
7. Salvamento

Esta padronização é uma peça fundamental para um projeto **clean code.** Tanto que esta introdução estará presente em todos os notebooks de todas as ferramentas do envelope de risco.

---


## 1. Importação do dataset e bibliotecas


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import genpareto

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"


## 2. Execução do `setup()` e alinhamento temporal


In [2]:
raw_df = pd.read_csv(CSV_PATH)

raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

# âncora temporal (já alinhada)
epf = pd.read_parquet("C:/projects/Libellula/data/processed/epf")

# alinhar EVT ao mesmo recorte
df = raw_df.loc[epf.index]

returns = df["Price"].pct_change()

## 3. Construção da ferramenta de risco


In [3]:
window = 100
threshold_q = 0.95
alpha = 0.99


def fit_evt(series, threshold_q):
    threshold = np.quantile(series, threshold_q)
    excesses = series[series > threshold] - threshold

    if len(excesses) < 5:
        return np.nan, np.nan

    c, loc, scale = genpareto.fit(excesses)
    return c, scale


evt_shape = []
evt_scale = []
evt_var = []
evt_cvar = []

for i in range(len(returns)):
    if i < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    window_data = returns.iloc[i - window : i].dropna()

    if len(window_data) < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    xi, beta = fit_evt(window_data, threshold_q)

    evt_shape.append(xi)
    evt_scale.append(beta)

    if np.isnan(xi) or np.isnan(beta):
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    threshold = np.quantile(window_data, threshold_q)

    var_evt = threshold + (beta / xi) * ((1 - alpha) ** (-xi) - 1)
    cvar_evt = (var_evt + (beta - xi * threshold)) / (1 - xi)

    evt_var.append(var_evt)
    evt_cvar.append(cvar_evt)

evt_df_check = pd.DataFrame(
    {
        "evt_shape": evt_shape,
        "evt_scale": evt_scale,
        "evt_var": evt_var,
        "evt_cvar": evt_cvar,
    },
    index=df.index,
)


## 4. Sanity checks mínimos


In [4]:
print("NaN críticos:")
print(evt_df_check.isna().sum())

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", evt_df_check.index.is_monotonic_increasing)
print("Index has duplicates:", evt_df_check.index.has_duplicates)

print("\nDistribuição plausível:")
print(evt_df_check.describe().T[["mean", "std", "min", "max"]])


NaN críticos:
evt_shape    101
evt_scale    101
evt_var      101
evt_cvar     101
dtype: int64

Alinhamento temporal:
Index monotonic increasing: True
Index has duplicates: False

Distribuição plausível:
               mean       std       min        max
evt_shape  1.341687  0.143546  0.452616   1.423964
evt_scale  0.000701  0.000514  0.000203   0.001803
evt_var    0.251995  0.165577  0.022298   0.648811
evt_cvar   0.704610  4.854992 -1.820264  17.357655


## 5. DataFrame final (features de risco)


In [5]:
evt_dataset = pd.DataFrame(index=df.index)

evt_dataset["evt_shape"] = evt_shape
evt_dataset["evt_scale"] = evt_scale
evt_dataset["evt_var"] = evt_var
evt_dataset["evt_cvar"] = evt_cvar


## 6. Padronização


In [6]:
evt_dataset = evt_dataset.sort_index().bfill()
evt_dataset.index.name = "timestamp"
evt_dataset = evt_dataset.astype(np.float32)

evt_dataset.tail()


,evt_shape,evt_scale,evt_var,evt_cvar
timestamp,,,,
2024-07-17,1.393293,0.000457,0.205143,-0.505447
2024-07-18,1.393293,0.000457,0.205143,-0.505447
2024-07-19,1.393293,0.000457,0.205143,-0.505447
2024-07-22,1.393293,0.000457,0.205143,-0.505447
2024-07-23,1.393293,0.000457,0.205143,-0.505447


## 7. Salvamento


In [7]:
OUTPUT_PATH = "../../data/processed/evt/evt_features.parquet"
evt_dataset.to_parquet(OUTPUT_PATH, index=True)
OUTPUT_PATH



'../../data/processed/evt/evt_features.parquet'